# 🎯 f6_m00c_export_probs.ipynb
## Añade la columna `prob_abandono` a `meta_test_app.parquet`

🎯 **Qué hace**: Calcula la probabilidad de abandono predicha por el modelo ganador (Stacking__balanced) para cada alumno del conjunto de test y la añade como columna `prob_abandono` al parquet unificado que usa la app Streamlit.

📋 **Requisitos**:
- `data/06_evaluacion/meta_test_app.parquet` (generado en `f6_m00b_preparacion_app`)
- `data/05_modelado/X_test_prep.parquet` (generado en Fase 5 — ya preprocesado, listo para el modelo)
- `data/05_modelado/models/Stacking__balanced.pkl` (modelo entrenado)

📤 **Genera**:
- `data/06_evaluacion/meta_test_app.parquet` — mismo fichero sobreescrito con la columna `prob_abandono` añadida (6.725 × 35 cols)

🔄 **Flujo**: `f6_m00_preparacion` → `f6_m00b_preparacion_app` → **`f6_m00c_export_probs`** → `loaders.py`

⚠️ **Importante**: Este notebook DEBE ejecutarse después de `f6_m00b_preparacion_app`. Si se vuelve a ejecutar `m00b`, regenera el parquet desde cero y hay que volver a pasar `m00c` para recuperar la columna `prob_abandono`. La celda 2 comprueba si la columna ya existe y avisa.

💡 **Enfoque**: Replicamos la lógica ya usada en `app/pages/p02_titulacion.py::_cargar_datos_app()` — usamos `X_test_prep.parquet` (ya preprocesado en Fase 5) directamente contra el modelo, sin volver a aplicar el pipeline. Así garantizamos que las probabilidades son idénticas a las que calcula la app en tiempo real en el resto de páginas.

➡️ **Siguiente**: La app Streamlit (Fase 7) usa `prob_abandono` precalculada en `pronostico_shared.py` para el box plot + rug plot de la comparativa de titulaciones y para la sección `¿Dónde estás respecto a otros alumnos?`.

In [1]:
# ── Celda 1: setup ────────────────────────────────────────────────────────────
# ROOT detectado subiendo niveles hasta encontrar src/ (nunca hardcodeado).
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
Python: 3.11.14


In [2]:
# ── Celda 2: comprobación de estado del parquet ──────────────────────────────
# Si `prob_abandono` ya existe, avisa (el notebook sobreescribe igualmente,
# pero así el usuario sabe que está reprocesando).
RUTA_META_APP = ROOT / "data" / "06_evaluacion" / "meta_test_app.parquet"

assert RUTA_META_APP.exists(), (
    f"❌ No encontrado: {RUTA_META_APP}\n"
    "   Ejecuta primero f6_m00b_preparacion_app.ipynb"
)

df_app = pd.read_parquet(RUTA_META_APP)
print(f"meta_test_app: {df_app.shape}")
print(f"Columnas actuales: {len(df_app.columns)}")

if "prob_abandono" in df_app.columns:
    print("\n⚠️ La columna 'prob_abandono' YA existe — se recalculará y sobrescribirá.")
    print(f"   Valores actuales: min={df_app['prob_abandono'].min():.4f}, "
          f"max={df_app['prob_abandono'].max():.4f}, "
          f"media={df_app['prob_abandono'].mean():.4f}")
else:
    print("\n✅ La columna 'prob_abandono' NO existe — se añadirá.")

meta_test_app: (6725, 34)
Columnas actuales: 34

✅ La columna 'prob_abandono' NO existe — se añadirá.


In [3]:
# ── Celda 3: cargar X_test_prep + modelo ──────────────────────────────────────
# X_test_prep.parquet contiene los 6.725 alumnos de test YA PREPROCESADOS
# por el pipeline de Fase 5 (imputación, escalado, codificación aplicados).
# Se puede pasar directamente a predict_proba sin transformaciones extra.
# Es la misma lógica que usa p02_titulacion._cargar_datos_app() (línea 170).
RUTA_X_PREP = ROOT / "data" / "05_modelado" / "X_test_prep.parquet"
RUTA_MODELO = ROOT / "data" / "05_modelado" / "models" / "Stacking__balanced.pkl"

for ruta, nombre in [(RUTA_X_PREP, "X_test_prep.parquet"),
                     (RUTA_MODELO, "Stacking__balanced.pkl")]:
    assert ruta.exists(), f"❌ No encontrado: {nombre} en {ruta}"

X_prep = pd.read_parquet(RUTA_X_PREP)
modelo = joblib.load(RUTA_MODELO)

print(f"✅ X_test_prep cargado: {X_prep.shape}")
print(f"   Features ({len(X_prep.columns)}): {list(X_prep.columns)}")
print(f"\n✅ Modelo cargado: {type(modelo).__name__}")

# Verificación: el número de alumnos debe coincidir con meta_test_app
assert len(X_prep) == len(df_app), (
    f"❌ Mismatch: X_prep tiene {len(X_prep)} filas, "
    f"df_app tiene {len(df_app)}"
)

# Verificación: índices idénticos (requisito para el merge final)
assert list(X_prep.index) == list(df_app.index), (
    "❌ Los índices de X_prep y df_app NO coinciden — "
    "re-ejecuta f6_m00_preparacion.ipynb y f6_m00b_preparacion_app.ipynb"
)
print("✅ Índices coinciden entre X_prep y df_app")

✅ X_test_prep cargado: (6725, 27)
   Features (27): ['cred_superados_anio_1er', 'cupo', 'pais_nombre', 'provincia', 'universidad_origen', 'edad_entrada', 'anios_gap', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'via_acceso', 'rama', 'n_anios_beca', 'anios_sin_beca', 'situacion_laboral', 'n_anios_trabajando', 'max_pagos', 'orden_preferencia', 'cred_repetidos', 'tasa_repeticion', 'n_anios_sin_notas', 'tasa_abandono_titulacion', 'sexo', 'indicador_interrupcion', 'nota_1er_anio_missing', 'nota_acceso_missing', 'nota_selectividad_missing']

✅ Modelo cargado: Pipeline
✅ Índices coinciden entre X_prep y df_app


In [4]:
# ── Celda 4: calcular prob_abandono ───────────────────────────────────────────
# Llamada directa a predict_proba con X_test_prep. Devuelve una matriz
# de N x 2 (prob clase 0, prob clase 1). Nos quedamos con la columna 1
# (probabilidad de abandono = clase positiva).
probs = modelo.predict_proba(X_prep)[:, 1]

print(f"✅ Probabilidades calculadas: {len(probs)} valores")
print(f"   Rango: [{probs.min():.4f}, {probs.max():.4f}]")
print(f"   Media: {probs.mean():.4f}   Mediana: {np.median(probs):.4f}")
print(f"   Std:   {probs.std():.4f}")

# Verificación de calibración: la media de probs debe estar cerca de la
# tasa real de abandono del test (modelo bien calibrado ≈ media real).
tasa_real = df_app["abandono"].mean()
print(f"\n📊 Tasa real de abandono en test: {tasa_real:.4f}")
print(f"   Media de probs predichas:       {probs.mean():.4f}")
print(f"   Diferencia:                     {abs(probs.mean() - tasa_real):.4f}")

# Distribución por niveles de riesgo (umbrales de la app)
n_bajo  = int((probs < 0.30).sum())
n_medio = int(((probs >= 0.30) & (probs < 0.60)).sum())
n_alto  = int((probs >= 0.60).sum())
print(f"\n📊 Distribución por niveles de riesgo:")
print(f"   Bajo  (< 30%):        {n_bajo:,}  ({n_bajo/len(probs)*100:.1f}%)".replace(",", "."))
print(f"   Medio (30-60%):       {n_medio:,}  ({n_medio/len(probs)*100:.1f}%)".replace(",", "."))
print(f"   Alto  (≥ 60%):        {n_alto:,}  ({n_alto/len(probs)*100:.1f}%)".replace(",", "."))

✅ Probabilidades calculadas: 6725 valores
   Rango: [0.0169, 0.9776]
   Media: 0.2974   Mediana: 0.0536
   Std:   0.3716

📊 Tasa real de abandono en test: 0.2925
   Media de probs predichas:       0.2974
   Diferencia:                     0.0049

📊 Distribución por niveles de riesgo:
   Bajo  (< 30%):        4.552  (67.7%)
   Medio (30-60%):       428  (6.4%)
   Alto  (≥ 60%):        1.745  (25.9%)


In [5]:
# ── Celda 5: verificación con casos canónicos ─────────────────────────────────
# Los mismos 5 casos que usa m00b, pero ahora comprobamos la prob predicha.
# Casos con abandono=1 deberían tener prob alta; los de abandono=0 prob baja.
CASOS_CANONICOS = {
    "C1 — Ing. Informática (FP, abandona)":          15872,
    "C2 — Medicina (nota 11.94, no abandona)":       11906,
    "C3 — Comunicación (nota alta, abandona)":        14957,
    "C4 — Ing. Informática (mujer, no abandona)":     32472,
    "C5 — Derecho (mayor 25, trabaja, abandona)":      7176,
}

# Serie con las probs indexadas como X_prep (y por tanto como df_app)
probs_serie = pd.Series(probs, index=X_prep.index, name="prob_abandono")

print("Caso                                            | Real | Prob predicha")
print("-" * 78)
n_coherentes = 0
for nombre, idx in CASOS_CANONICOS.items():
    real = df_app.loc[idx, "abandono"]
    prob = probs_serie.loc[idx]
    icono = "🔴" if real == 1 else "🟢"
    coherente = (real == 1 and prob >= 0.5) or (real == 0 and prob < 0.5)
    check = "✅" if coherente else "⚠️"
    if coherente:
        n_coherentes += 1
    print(f"{icono} {nombre:<45}|  {int(real)}   | {prob:.4f} {check}")

print("-" * 78)
print(f"Casos coherentes: {n_coherentes}/5")
if n_coherentes == 5:
    print("✅ Todos los casos canónicos son coherentes con el abandono real")
else:
    print("⚠️ Algunos casos son incoherentes — revisar si es esperable (ej: perfiles ambiguos)")

Caso                                            | Real | Prob predicha
------------------------------------------------------------------------------
🔴 C1 — Ing. Informática (FP, abandona)         |  1   | 0.5672 ✅
🟢 C2 — Medicina (nota 11.94, no abandona)      |  0   | 0.0227 ✅
🔴 C3 — Comunicación (nota alta, abandona)      |  1   | 0.0421 ⚠️
🟢 C4 — Ing. Informática (mujer, no abandona)   |  0   | 0.4914 ✅
🔴 C5 — Derecho (mayor 25, trabaja, abandona)   |  1   | 0.5321 ✅
------------------------------------------------------------------------------
Casos coherentes: 4/5
⚠️ Algunos casos son incoherentes — revisar si es esperable (ej: perfiles ambiguos)


In [6]:
# ── Celda 6: añadir columna y guardar ─────────────────────────────────────────
# Sobrescribimos meta_test_app.parquet con la nueva columna prob_abandono.
# Así loaders.py lo cargará sin cambios.
df_app["prob_abandono"] = probs_serie

# Verificación final: sin nulos en prob_abandono y shape esperado
assert df_app["prob_abandono"].isnull().sum() == 0, "❌ NaN en prob_abandono"
assert df_app.shape[0] == len(probs), "❌ Longitudes inconsistentes"

df_app.to_parquet(RUTA_META_APP, index=True)

print(f"✅ Guardado: {RUTA_META_APP}")
print(f"   Shape:   {df_app.shape}")
print(f"   Tamaño:  {RUTA_META_APP.stat().st_size / 1024:.1f} KB")
print(f"\n🎯 Columna prob_abandono lista para usar en la app:")
print(f"   Rango:   [{df_app['prob_abandono'].min():.4f}, {df_app['prob_abandono'].max():.4f}]")
print(f"   Media:   {df_app['prob_abandono'].mean():.4f}")

✅ Guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\06_evaluacion\meta_test_app.parquet
   Shape:   (6725, 35)
   Tamaño:  276.0 KB

🎯 Columna prob_abandono lista para usar en la app:
   Rango:   [0.0169, 0.9776]
   Media:   0.2974
